<table>
  <tr>
    <td><div align="left"><font size="30">System linearization</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True)  # create simulator
bd = sim.blockdiagram()  # create an empty block diagram

M = 1.0  # mass of cart
m = 0.1  # mass of pendulum
l = 0.5  # length of pendulum
g = -9.81  # gravity
I = m * l**2 / 3  # inertia of pendulum about CoG  ????
b = 10  # cart friction

x_dot = bd.INTEGRATOR(name="x_dot")
x = x_dot >> bd.INTEGRATOR(name="x")

theta_dot = bd.INTEGRATOR(name="theta_dot")
theta = theta_dot >> bd.INTEGRATOR(name="theta")

cos_theta = theta >> bd.FUNCTION(math.cos, name="cos(theta)")
sin_theta = theta >> bd.FUNCTION(math.sin, name="sin(theta)")

K = np.zeros((1,4))
F = bd.PIECEWISE((0,0), (1,1), (1.2,0), name="disturbance") -  K * bd.STATE()

D = M + m * sin_theta**2  # determinant of the mass matrix
f = F - b * x_dot  # nett force on the cart (input minus friction)

x_dot[0] = (
    f + m * sin_theta * (l * theta_dot**2 + g * cos_theta)
) / D  # cart acceleration
theta_dot[0] = (
    -f * cos_theta
    - m * l * theta_dot**2 * cos_theta * sin_theta
    - (M + m) * g * sin_theta
) / (l * D)  # pendulum angular acceleration

T = (
    1 / 2 * (M + m) * x_dot**2
    + 1 / 2 * m * l**2 * theta_dot**2
    + m * l * x_dot * theta_dot * cos_theta
)  # kinetic energy (note coupling term from cart-pole interaction)
U = -m * g * l * (1+cos_theta)  # potential energy (zero at vertical down)
E = T + U

# display the system state evolution on a scope and animation
scope = bd.SCOPE(
    nin=4,
    inputs=[F, x, theta, E],
    labels=("F", "x", r"$\theta$", "E"),
    loc="lower right",
)

Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

In [ ]:
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 5s
print(out)


In [ ]:
X = bd.linearize(t= 0, inputs=["disturbance"], outputs=[scope[1], scope[2]], full=True)
print(X)



Now we can design a linear controller using the Python Control package

```
$ pip install control
```

In [ ]:
import control as ct

# Create the state-space system (optional, but good practice)
sys = ct.ss(X.A, X.B, X.C, X.D, states=X.xindex.names, inputs=X.uindex.names, outputs=X.yindex.names)

# 2. Define the cost matrices
# Q must be positive semi-definite (penalizes state errors)
# R must be positive definite (penalizes control effort)
Q = np.diag([1, 1, 10, 1])  # weights for [x, x_dot, theta, theta_dot]
R = np.array([[100]]) # weight for control input (force F)

# 3. Compute the LQR optimal gain matrix K
# control.lqr returns:
# K: The optimal feedback gain matrix (u = -Kx)
# S: The solution to the algebraic Riccati equation
# E: The eigenvalues (poles) of the closed-loop system (A - BK)
K, S, E = ct.lqr(sys, Q, R)

# Note: You can also pass the arrays directly without creating a sys object:
# K, S, E = ct.lqr(A, B, Q, R)

print("Optimal Feedback Gain (K): \n", K)
print("Closed-loop Poles (E): \n", E)

In [ ]:
bd["_gain.0"].set_param("K", K)  # set the gain block to the computed LQR gain
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 5s

In [ ]:
bd.blocknames
